# Cruce de Graduados USTA con el **RUES** (trayectoria emprendedora)

**Objetivo.** Identificar cuáles de los graduados (dataset consolidado en
`salidas/graduados_integrado.csv`) figuran en el **RUES** como titulares de una
matrícula mercantil —es decir, **han creado o registrado empresa**— y caracterizar
esa actividad: número de matrículas, vigencia (activa/cancelada), sector y antigüedad.

## Fuente: RUES (Registro Único Empresarial y Social)

- Administrado por **Confecámaras** / las Cámaras de Comercio.
- Conjunto en Datos Abiertos: **«Personas Naturales, Personas Jurídicas y Entidades
  Sin Ánimo de Lucro»** — Socrata **`c82u-588k`**
  (el mismo identificado en el proyecto `ustadistica/impacto_graduados`, fuente RUES).
- Tamaño: **~9,3 millones** de matrículas. El documento del titular está en la columna
  **`numero_identificacion`** (para persona natural es la cédula).

## Estrategia (igual que el cruce con SECOP)

Tenemos ~174 685 cédulas únicas. En lugar de descargar los 9,3M de registros, el cruce
se hace **del lado del servidor**: por lotes se envía
`numero_identificacion IN (lista_de_cedulas)` agrupado por documento, de modo que la
API **solo devuelve las coincidencias** ya agregadas (n.º de matrículas, cuántas
activas, razón social, sector y fechas).

> **Lectura del RUES (impacto emprendedor).** Una coincidencia indica que el egresado
> aparece como titular de una matrícula mercantil. El estado `ACTIVA` vs `CANCELADA`
> permite aproximar la **supervivencia** del emprendimiento.


## 1. Configuración e importaciones

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

API_URL  = "https://www.datos.gov.co/resource/c82u-588k.json"  # RUES
COL_DOC  = "numero_identificacion"
TAM_LOTE = 250
PAUSA    = 0.10
TIMEOUT  = 120

APP_TOKEN = None        # token Socrata opcional (sube el cupo de peticiones)
SESSION = requests.Session()
if APP_TOKEN:
    SESSION.headers.update({"X-App-Token": APP_TOKEN})

SALIDA = Path("salidas"); SALIDA.mkdir(exist_ok=True)
print("API:", API_URL)


## 2. Cédulas a consultar

Se reutilizan las identificaciones del dataset integrado (solo dígitos, documentos
plausibles).


In [ ]:
g = pd.read_csv(SALIDA / "graduados_integrado.csv",
                usecols=["identificacion", "fuente"], dtype={"identificacion": "string"})
ced = g["identificacion"].dropna().str.strip()
ced = ced[ced.str.fullmatch(r"\d{4,12}")]
cedulas_unicas = sorted(ced.unique())
print(f"Cédulas únicas a consultar: {len(cedulas_unicas):,}")


## 3. Función de consulta por lotes (SoQL)

Cada petición agrupa por `numero_identificacion`. Se usa `case(...)` del lado del
servidor para contar matrículas **activas** sin traer fila por fila. Incluye
reintentos con *backoff*.


In [ ]:
SELECT = (
    "numero_identificacion,"
    "count(1) as n_matriculas,"
    "sum(case(estado_matricula='ACTIVA',1,true,0)) as n_activas,"
    "max(razon_social) as razon_social,"
    "max(categoria_matricula) as categoria,"
    "max(camara_comercio) as camara_comercio,"
    "max(cod_ciiu_act_econ_pri) as ciiu_principal,"
    "min(fecha_matricula) as primera_matricula,"
    "max(fecha_matricula) as ultima_matricula"
)

def consultar_lote(cedulas, intentos=4):
    '''Consulta un lote de cédulas en el RUES. Devuelve una fila por cédula que
    SÍ aparece con matrícula mercantil.'''
    in_list = ",".join("'%s'" % c for c in cedulas)
    params = {
        "$select": SELECT,
        "$where": f"{COL_DOC} in ({in_list})",
        "$group": COL_DOC,
        "$limit": 50000,
    }
    for k in range(intentos):
        try:
            r = SESSION.get(API_URL, params=params, timeout=TIMEOUT)
            r.raise_for_status()
            return r.json()
        except Exception:
            if k == intentos - 1:
                raise
            time.sleep(2 ** k)
    return []

def lotes(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


## 4. Ejecución del cruce

In [ ]:
# --- Caché-primero: si ya existe el dataset de matriculados, se omite la consulta a la API ---
_cache_rues = SALIDA / "graduados_emprendedores_rues.csv"
USAR_CACHE = _cache_rues.exists()
if USAR_CACHE:
    rues = pd.read_csv(_cache_rues, dtype={"identificacion": "string"})[
        ["identificacion", "n_matriculas", "n_activas", "razon_social", "categoria",
         "camara_comercio", "ciiu_principal", "primera_matricula", "ultima_matricula",
         "tiene_empresa_activa"]].copy()
    print(f"Cache encontrada ({_cache_rues.name}): {len(rues):,} matriculados. Se omite la consulta a la API.")
    resultados = None
else:
    resultados = []
    total_lotes = (len(cedulas_unicas) + TAM_LOTE - 1) // TAM_LOTE
    t0 = time.time()
    for i, lote in enumerate(lotes(cedulas_unicas, TAM_LOTE), start=1):
        resultados.extend(consultar_lote(lote))
        if i % 20 == 0 or i == total_lotes:
            print(f"Lote {i:>3}/{total_lotes} | coincidencias acumuladas: {len(resultados):>5} | {time.time()-t0:5.0f}s")
        time.sleep(PAUSA)
    print(f"\nListo. Graduados con matrícula en RUES: {len(resultados):,} en {time.time()-t0:.0f}s")


## 5. Construcción de la tabla de emprendedores

Se tipan los agregados. Las fechas del RUES vienen como texto `AAAAMMDD`, se convierten
a `datetime`.


In [ ]:
if resultados is not None:                      # los datos vinieron de la API
    rues = pd.DataFrame(resultados)
    if rues.empty:
        raise RuntimeError("No se obtuvieron resultados; revisa la conexión o la API.")
    rues = rues.rename(columns={"numero_identificacion": "identificacion"})
    def fecha_aaaammdd(s):
        return pd.to_datetime(s, format="%Y%m%d", errors="coerce")
    rues["primera_matricula"] = fecha_aaaammdd(rues["primera_matricula"])
    rues["ultima_matricula"]  = fecha_aaaammdd(rues["ultima_matricula"])
else:                                           # los datos vinieron de la caché (fechas en ISO)
    rues["primera_matricula"] = pd.to_datetime(rues["primera_matricula"], errors="coerce")
    rues["ultima_matricula"]  = pd.to_datetime(rues["ultima_matricula"], errors="coerce")

rues["identificacion"] = rues["identificacion"].astype("string")
rues["n_matriculas"]   = pd.to_numeric(rues["n_matriculas"], errors="coerce").astype("Int64")
rues["n_activas"]      = pd.to_numeric(rues["n_activas"], errors="coerce").astype("Int64")
rues["ciiu_principal"] = rues["ciiu_principal"].astype("string")
rues["tiene_empresa_activa"] = rues["n_activas"].fillna(0) > 0
rues = rues.drop_duplicates("identificacion").sort_values("n_matriculas", ascending=False)
print("Cédulas con matrícula en RUES:", rues["identificacion"].nunique())
rues.head(10)


## 6. Marcar y enriquecer el dataset de graduados

In [ ]:
grad = pd.read_csv(SALIDA / "graduados_integrado.csv", dtype={"identificacion": "string"})
grad["identificacion"] = grad["identificacion"].str.strip()

cruce = grad.merge(rues, on="identificacion", how="left")
cruce["en_rues"] = cruce["n_matriculas"].notna()

n_reg = len(cruce)
ced_total = cruce["identificacion"].dropna().nunique()
ced_rues  = cruce.loc[cruce["en_rues"], "identificacion"].nunique()
ced_activa = cruce.loc[cruce["tiene_empresa_activa"] == True, "identificacion"].nunique()

print(f"Cédulas únicas de graduados        : {ced_total:,}")
print(f"  -> con matrícula en RUES         : {ced_rues:,} ({ced_rues/ced_total*100:.1f}%)")
print(f"  -> con al menos una empresa ACTIVA: {ced_activa:,} ({ced_activa/ced_total*100:.1f}%)")


## 7. Análisis

### 7.1 Tasa de emprendimiento por fuente


In [ ]:
tmp = cruce.dropna(subset=["identificacion"]).copy()
grad_x   = tmp.groupby("fuente")["identificacion"].nunique()
rues_x   = tmp[tmp["en_rues"]].groupby("fuente")["identificacion"].nunique()
activa_x = tmp[tmp["tiene_empresa_activa"] == True].groupby("fuente")["identificacion"].nunique()
por_fuente = pd.DataFrame({"graduados": grad_x, "en_rues": rues_x, "empresa_activa": activa_x}).fillna(0).astype(int)
por_fuente["pct_en_rues"] = (por_fuente["en_rues"] / por_fuente["graduados"] * 100).round(1)
por_fuente["pct_activa"]  = (por_fuente["empresa_activa"] / por_fuente["graduados"] * 100).round(1)
display(por_fuente)

ax = por_fuente[["pct_en_rues", "pct_activa"]].plot(kind="bar")
ax.set_title("% de graduados con matrícula RUES y con empresa activa, por fuente")
ax.set_ylabel("%"); ax.set_xlabel("Fuente"); ax.legend(["Con matrícula", "Empresa activa"])
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


### 7.2 Supervivencia (activas vs canceladas)

In [ ]:
n_emp = int(rues["en_rues"].sum()) if "en_rues" in rues else len(rues)
activas = int((rues["tiene_empresa_activa"] == True).sum())
canceladas = len(rues) - activas
print(f"Graduados-emprendedores            : {len(rues):,}")
print(f"  con empresa activa hoy           : {activas:,} ({activas/len(rues)*100:.1f}%)")
print(f"  sin empresa activa (cancelada)   : {canceladas:,} ({canceladas/len(rues)*100:.1f}%)")

ax = pd.Series({"Activa": activas, "Cancelada/otro": canceladas}).plot(
    kind="pie", autopct="%1.1f%%", colors=["#1B998B", "#E07A5F"], ylabel="")
ax.set_title("Estado de la matrícula de los graduados-emprendedores")
plt.tight_layout(); plt.show()


### 7.3 Top de programas por número de graduados-emprendedores

In [ ]:
emp_prog = (cruce[cruce["en_rues"]]
            .dropna(subset=["identificacion"])
            .groupby("programa_norm")["identificacion"].nunique()
            .sort_values(ascending=False).head(15))
ax = emp_prog.sort_values().plot(kind="barh", color="#274690")
ax.set_title("Top 15 programas por n.º de graduados con matrícula en RUES")
ax.set_xlabel("Cédulas únicas con empresa"); ax.set_ylabel("")
plt.tight_layout(); plt.show()
emp_prog.to_frame("emprendedores")


### 7.4 Sectores económicos (CIIU principal) más frecuentes

In [ ]:
top_ciiu = rues["ciiu_principal"].value_counts().head(15)
ax = top_ciiu.sort_values().plot(kind="barh", color="#5C415D")
ax.set_title("Top 15 códigos CIIU (actividad económica principal) de los emprendedores")
ax.set_xlabel("N.º de graduados-emprendedores"); ax.set_ylabel("Código CIIU")
plt.tight_layout(); plt.show()
top_ciiu.to_frame("n")


## 7.5 Análisis ampliado: organización jurídica, sector y supervivencia

Enriquecemos el cruce con dimensiones que el RUES expone por matrícula: la **organización
jurídica** (tipo de empresa), el **estado** de la matrícula (supervivencia), el **sector
económico** (CIIU) cruzado con la carrera, y la supervivencia por carrera y cohorte. La
extracción usa el patrón **cargar-o-extraer**: si `salidas/rues_dims.csv` existe se carga;
si no, se consulta la API por lotes y se guarda.

In [ ]:
import time, re, unicodedata
from pathlib import Path
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt

SAL = Path("salidas")
API_RUES = "https://www.datos.gov.co/resource/c82u-588k.json"
SES = requests.Session()

gi = pd.read_csv(SAL / "graduados_integrado.csv", dtype={"identificacion": "string"})
_c = gi["identificacion"].dropna().str.strip()
CEDULAS = sorted(_c[_c.str.fullmatch(r"\d{4,12}")].unique())
ced_prog = gi.dropna(subset=["identificacion"]).groupby("identificacion")["programa_norm"].agg(lambda s: s.value_counts().index[0])
ced_sec  = gi.dropna(subset=["identificacion"]).groupby("identificacion")["fuente"].first()
SECC = ["General", "Tunja", "Villavicencio"]
SECC_LBL = {"General": "Bucaramanga", "Tunja": "Tunja", "Villavicencio": "Villavicencio"}

def sector_ciiu(code):
    '''Mapea un código CIIU a su sección/sector económico legible.'''
    s = re.sub(r"\D", "", str(code))
    if len(s) < 2: return "(sin dato)"
    d = int(s[:2])
    rangos = [((1,3),"Agropecuario"),((5,9),"Minería"),((10,33),"Industria manufacturera"),
              ((35,35),"Energía"),((36,39),"Agua y saneamiento"),((41,43),"Construcción"),
              ((45,47),"Comercio"),((49,53),"Transporte y logística"),((55,56),"Alojamiento y comida"),
              ((58,63),"Información y comunicaciones"),((64,66),"Financieras y seguros"),((68,68),"Inmobiliarias"),
              ((69,75),"Profesionales y técnicas"),((77,82),"Servicios administrativos"),((84,84),"Administración pública"),
              ((85,85),"Educación"),((86,88),"Salud y asistencia social"),((90,93),"Arte y entretenimiento"),
              ((94,96),"Otros servicios")]
    for (a,b),lab in rangos:
        if a <= d <= b: return lab
    return "(sin/otro)"

def consultar_rues(select, group, tam=300, intentos=5):
    '''Consulta agregada por lotes de cédulas contra el RUES (numero_identificacion).'''
    res = []
    for i in range(0, len(CEDULAS), tam):
        inlist = ",".join("'%s'" % c for c in CEDULAS[i:i+tam])
        params = {"$select": select, "$where": f"numero_identificacion in ({inlist})",
                  "$group": group, "$limit": 50000}
        for k in range(intentos):
            try:
                r = SES.get(API_RUES, params=params, timeout=180); r.raise_for_status()
                res.extend(r.json()); break
            except Exception:
                if k == intentos - 1: raise
                time.sleep(2 ** k)
        time.sleep(0.05)
    return pd.DataFrame(res)

print("Cédulas:", f"{len(CEDULAS):,}")

In [ ]:
f = SAL / "rues_dims.csv"
if f.exists():
    dims = pd.read_csv(f, dtype={"identificacion": "string"})
    print("Cargado de caché:", f.name)
else:
    dims = consultar_rues(
        "numero_identificacion, categoria_matricula, organizacion_juridica, estado_matricula, cod_ciiu_act_econ_pri, count(1) as n",
        "numero_identificacion, categoria_matricula, organizacion_juridica, estado_matricula, cod_ciiu_act_econ_pri")
    dims = dims.rename(columns={"numero_identificacion": "identificacion", "cod_ciiu_act_econ_pri": "ciiu"})
    dims["n"] = pd.to_numeric(dims["n"], errors="coerce")
    dims.to_csv(f, index=False, encoding="utf-8-sig")
    print("Extraído y guardado:", f.name)
print("Filas:", f"{len(dims):,}")

# Organización jurídica (tipo de empresa) y estado (supervivencia)
org = dims.groupby("organizacion_juridica")["n"].sum().sort_values(ascending=False)
print("\nOrganización jurídica (% de matrículas):")
display((org / org.sum() * 100).round(1).head(6).to_frame("%"))
est = dims.groupby("estado_matricula")["n"].sum().sort_values(ascending=False)
print("Estado de la matrícula (% — supervivencia):")
display((est / est.sum() * 100).round(1).head(6).to_frame("%"))

In [ ]:
# Sector económico por carrera (firma sectorial)
emp = pd.read_csv(SAL / "graduados_emprendedores_rues.csv", dtype={"identificacion": "string"})
emp["prog"] = emp["identificacion"].map(ced_prog)
emp["sector"] = emp["ciiu_principal"].map(sector_ciiu)
emp["activa"] = emp["tiene_empresa_activa"].astype(str).str.lower().isin(["true", "1", "verdadero"])

topprogs = emp["prog"].value_counts().head(8).index.tolist()
topsec = emp["sector"].value_counts().drop(labels=["(sin dato)", "(sin/otro)"], errors="ignore").head(7).index.tolist()
he = emp[emp["prog"].isin(topprogs) & emp["sector"].isin(topsec)]
pv = he.pivot_table(index="prog", columns="sector", values="identificacion", aggfunc="nunique", fill_value=0).reindex(topprogs)
pvp = (pv.div(pv.sum(axis=1).replace(0, np.nan), axis=0) * 100).round(0)

fig, ax = plt.subplots(figsize=(9, 4.2)); im = ax.imshow(pvp.values, cmap="Greens", aspect="auto")
ax.set_yticks(range(len(topprogs))); ax.set_yticklabels([str(p).title()[:22] for p in topprogs], fontsize=8)
ax.set_xticks(range(len(pvp.columns))); ax.set_xticklabels(pvp.columns, rotation=30, ha="right", fontsize=8)
for i in range(len(topprogs)):
    for j in range(len(pvp.columns)):
        v = pvp.values[i, j]
        if v == v: ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7, color="white" if v > 50 else "black")
fig.colorbar(im, label="% de emprendedores"); ax.set_title("Sector económico por carrera (RUES)")
plt.tight_layout(); plt.show()
print("El comercio domina casi todas las carreras; Odontología→Salud y Arquitectura→Profesionales son las excepciones.")

In [ ]:
# Supervivencia por carrera y por cohorte (antigüedad)
sup = emp.groupby("prog").agg(n=("identificacion", "nunique"), act=("activa", "mean"))
sup = sup[sup["n"] >= 80].sort_values("act", ascending=False)
print("Supervivencia (% activa) por carrera — top:")
display((sup["act"] * 100).round(1).head(10).to_frame("% activa"))

emp["anio_mat"] = pd.to_datetime(emp["primera_matricula"], errors="coerce").dt.year
co = emp.dropna(subset=["anio_mat"]); co = co[(co["anio_mat"] >= 1990) & (co["anio_mat"] <= 2024)]
coh = co.groupby((co["anio_mat"] // 5 * 5).astype(int)).agg(n=("identificacion", "nunique"), act=("activa", "mean"))
coh = coh[coh["n"] >= 50]
ax = (coh["act"] * 100).plot(marker="o", color="#1B998B", figsize=(7, 3.6))
ax.set_xlabel("Quinquenio de la primera matrícula"); ax.set_ylabel("% activa hoy")
ax.set_title("Supervivencia según antigüedad de la matrícula")
plt.tight_layout(); plt.show()
print("La supervivencia depende fuertemente de la antigüedad: las matrículas recientes siguen activas en mayor proporción.")

## 7.6 Dataset ampliado generado

El análisis anterior deja en `salidas/` el insumo `rues_dims.csv` ---organización jurídica,
estado y CIIU por cédula---, reutilizable para análisis de tipo de empresa, supervivencia y
sector económico.

## 8. Exportación

- `graduados_rues_cruce.csv`: dataset integrado + bandera y métricas RUES.
- `graduados_emprendedores_rues.csv`: solo las cédulas con matrícula mercantil.


In [ ]:
ruta_cruce = SALIDA / "graduados_rues_cruce.csv"
ruta_emp   = SALIDA / "graduados_emprendedores_rues.csv"
cruce.to_csv(ruta_cruce, index=False, encoding="utf-8-sig")

emp_export = (cruce[cruce["en_rues"]]
              .sort_values("n_matriculas", ascending=False)
              .drop_duplicates("identificacion")
              [["identificacion", "nombre_completo", "fuente", "sede", "programa",
                "razon_social", "categoria", "camara_comercio", "ciiu_principal",
                "n_matriculas", "n_activas", "tiene_empresa_activa",
                "primera_matricula", "ultima_matricula"]])
emp_export.to_csv(ruta_emp, index=False, encoding="utf-8-sig")
print("Exportado:")
print(" -", ruta_cruce.resolve())
print(" -", ruta_emp.resolve(), f"({len(emp_export):,} emprendedores)")


## 9. Conclusiones y limitaciones

**Resultado.** Se cruzaron las cédulas de los graduados contra los ~9,3M de matrículas
del RUES y se marcó quiénes figuran como titulares de empresa, su estado (activa /
cancelada), sector (CIIU) y antigüedad — una aproximación a la **trayectoria
emprendedora** del egresado.

**Limitaciones / advertencias**
- **Match por número de documento.** Se asume que `numero_identificacion` (cédula)
  corresponde a la misma persona graduada; la cédula es única, pero pueden existir
  errores de digitación.
- **Solo persona natural por cédula.** Empresas constituidas como **persona jurídica**
  (S.A.S., Ltda., etc.) figuran en el RUES con **NIT**, no con la cédula del socio; este
  cruce **no** las captura. Por tanto el resultado **subestima** el emprendimiento real
  (un egresado puede ser dueño de una SAS sin aparecer aquí por cédula).
- **Estado vigente.** `estado_matricula` refleja el estado actual; no reconstruye toda
  la historia de renovaciones.
- **Homónimos:** el cruce es por documento, no por nombre; se exporta `razon_social`
  para verificación manual.

> Nota: este cruce es complementario al de SECOP (contratación pública). Juntos
> describen dos facetas del impacto del egresado: **emprendedora** (RUES) y como
> **proveedor del Estado** (SECOP).
